In [51]:
import requests
import pandas as pd

API_KEY = 'f91688b2469810e18dbf6649b7d462fe'
sport_key = 'soccer_epl'
region = 'uk,eu'
markets = 'h2h,totals,spreads'
url = f"https://api.the-odds-api.com/v4/sports/{sport_key}/odds/?apiKey={API_KEY}&regions={region}&markets={markets}"

response = requests.get(url)
try:
    data = response.json()
except Exception as e:
    print("API did not return JSON:", response.text)
    raise e

if not isinstance(data, list):
    print("API returned unexpected result. Details:")
    print(data)
    print("Status code:", response.status_code)
    raise Exception("Odds API returned error. Check your API key or quota.")

all_rows = []
all_colnames = set()

for event in data:
    ev = {
        "date": event.get("commence_time"),
        "home_team": event.get("home_team"),
        "away_team": event.get("away_team")
    }
    for bookmaker in event.get("bookmakers", []):
        book = bookmaker.get("key", "")
        for market in bookmaker.get("markets", []):
            mkt = market.get("key", "")
            for out in market.get("outcomes", []):
                if mkt == "h2h":
                    if out["name"] == event["home_team"]:
                        col = f"{book}_H"
                    elif out["name"] == event["away_team"]:
                        col = f"{book}_A"
                    elif out["name"].lower() == "draw":
                        col = f"{book}_D"
                    else:
                        col = f"{book}_h2h_{out['name']}"
                    ev[col] = out["price"]
                    all_colnames.add(col)
                elif mkt == "totals":
                    sign = ">" if out["name"].lower() == "over" else "<"
                    pt = out.get("point", "")
                    col = f"{book}_{sign}{pt}"
                    ev[col] = out["price"]
                    all_colnames.add(col)
                elif mkt == "spreads":
                    side = "H" if out["name"] == event["home_team"] else "A"
                    pt = out.get("point", "")
                    col = f"{book}_AH{side}_{pt}"
                    ev[col] = out["price"]
                    all_colnames.add(col)
    all_rows.append(ev)

all_colnames = ["date", "home_team", "away_team"] + sorted(all_colnames)
odds_df = pd.DataFrame(all_rows)

for col in all_colnames:
    if col not in odds_df.columns:
        odds_df[col] = pd.NA

odds_df = odds_df[all_colnames]

odds_df.to_csv("all_epl_odds.csv", index=False)
print("\n✅ Built all_epl_odds.csv with these columns:")
print(odds_df.columns.tolist())
print(odds_df.head(10))


✅ Built all_epl_odds.csv with these columns:
['date', 'home_team', 'away_team', 'betclic_fr_A', 'betclic_fr_D', 'betclic_fr_H', 'betfair_ex_eu_A', 'betfair_ex_eu_D', 'betfair_ex_eu_H', 'betfair_ex_uk_A', 'betfair_ex_uk_D', 'betfair_ex_uk_H', 'betfair_sb_uk_A', 'betfair_sb_uk_D', 'betfair_sb_uk_H', 'betfred_uk_A', 'betfred_uk_D', 'betfred_uk_H', 'betonlineag_<2.5', 'betonlineag_<3.0', 'betonlineag_<3.5', 'betonlineag_>2.5', 'betonlineag_>3.0', 'betonlineag_>3.5', 'betonlineag_A', 'betonlineag_AHA_-0.5', 'betonlineag_AHA_-2.0', 'betonlineag_AHA_0.0', 'betonlineag_AHA_0.5', 'betonlineag_AHA_1.0', 'betonlineag_AHH_-0.5', 'betonlineag_AHH_-1.0', 'betonlineag_AHH_0.0', 'betonlineag_AHH_0.5', 'betonlineag_AHH_2.0', 'betonlineag_D', 'betonlineag_H', 'betsson_<2.5', 'betsson_>2.5', 'betsson_A', 'betsson_D', 'betsson_H', 'betvictor_A', 'betvictor_D', 'betvictor_H', 'betway_A', 'betway_D', 'betway_H', 'boylesports_A', 'boylesports_D', 'boylesports_H', 'casumo_<2.5', 'casumo_<3.5', 'casumo_>2.5',

In [52]:
# Uncomment to see your current quota at any time:
print("Requests remaining:", response.headers.get("x-requests-remaining"))
print("Requests used:", response.headers.get("x-requests-used"))
print("Requests cost for this call:", response.headers.get("x-requests-last"))

Requests remaining: 445
Requests used: 55
Requests cost for this call: 6
